# Obesity Level Classification - Model Training

Training and evaluating 5 classifiers on the UCI obesity dataset (run on BITS Virtual Lab).

## 1. Install requirements
Keep this notebook inside the project folder so the relative paths work.

In [ ]:
!pip install -r requirements.txt

## 2. Import libraries

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix, classification_report)

SEED = 2026
TARGET = "NObeyesdad"
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

## 3. Load the dataset

In [ ]:
df = pd.read_csv("data/raw/ObesityDataSet_raw_and_data_sinthetic.csv")
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
df.head()

In [ ]:
print(df[TARGET].value_counts())
df[TARGET].value_counts().plot(kind="bar", title="Class distribution")
plt.show()

## 4. Pre-processing

In [ ]:
X = df.drop(columns=[TARGET])
encoder = LabelEncoder()
y = encoder.fit_transform(df[TARGET])
print("Classes:", list(encoder.classes_))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

## 5. Evaluation function (6 metrics + confusion matrix)

In [ ]:
results = {}

def evaluate(name, pipe):
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)
    scores = {
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, pred),
    }
    results[name] = scores
    print(name)
    for k, v in scores.items():
        print(f"  {k}: {v:.4f}")

    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.title(f"Confusion matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.show()

    joblib.dump(pipe, MODEL_DIR / (name.lower().replace(" ", "_") + ".joblib"))

## 6. Model 1 - Logistic Regression

In [ ]:
lr = Pipeline([("prep", make_preprocessor()),
               ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
lr.fit(X_train, y_train)
evaluate("Logistic Regression", lr)

## 7. Model 2 - Decision Tree

In [ ]:
dt = Pipeline([("prep", make_preprocessor()),
               ("clf", DecisionTreeClassifier(max_depth=12, random_state=SEED))])
dt.fit(X_train, y_train)
evaluate("Decision Tree", dt)

## 8. Model 3 - K-Nearest Neighbors

In [ ]:
knn = Pipeline([("prep", make_preprocessor()),
                ("clf", KNeighborsClassifier(n_neighbors=7, weights="distance"))])
knn.fit(X_train, y_train)
evaluate("KNN", knn)

## 9. Model 4 - Gaussian Naive Bayes

In [ ]:
nb = Pipeline([("prep", make_preprocessor()),
               ("clf", GaussianNB())])
nb.fit(X_train, y_train)
evaluate("Naive Bayes", nb)

## 10. Model 5 - Support Vector Machine

In [ ]:
svm = Pipeline([("prep", make_preprocessor()),
                ("clf", SVC(kernel="rbf", C=5.0, probability=True, random_state=SEED))])
svm.fit(X_train, y_train)
evaluate("Support Vector Machine", svm)

## 11. Comparison of all 5 models

In [ ]:
table = pd.DataFrame(results).T.round(4)
table

In [ ]:
print(classification_report(y_test, svm.predict(X_test), target_names=encoder.classes_))

## 12. Save encoder and metrics

In [ ]:
joblib.dump(encoder, MODEL_DIR / "label_encoder.joblib")
(MODEL_DIR / "metrics.json").write_text(json.dumps(table.to_dict(orient="index"), indent=2))
!ls -l model